Import Libs

In [ ]:
import os
from pathlib import Path
from docutils.nodes import math
from mpi4py import MPI
from petsc4py import PETSc
import math
import pyvista

import numpy as np

import ufl
import dolfinx
from basix.ufl import element, mixed_element
from dolfinx import default_real_type, log, plot
from dolfinx.fem import Function, functionspace
from dolfinx.fem.petsc import NonlinearProblem
from dolfinx.io import XDMFFile
from dolfinx.mesh import CellType, create_unit_square, create_interval
from ufl import TestFunction, TestFunctions

try:
    import pyvista as pv
    import pyvistaqt as pvqt

    have_pyvista = True
    print("load successfully")
except ModuleNotFoundError:
    print("pyvista and pyvistaqt are required to visualise the solution")
    have_pyvista = False


# Save all logging to file
log.set_output_file("log.txt")

Define Constants

In [ ]:
c_max = 30000
L = 11e-6 # micrometer
omega_a_tilde = 64.3e-3 / (8.617e-5 * 298)
omega_b_tilde = 23.1e-3 / (8.617e-5 * 298)
omega_c_tilde = 4.1e-3 / (8.617e-5 * 298)
T = 298
T_ref = 298
mu_ref = 0
k = 6.7e-7
D = 1.26e-12
i_0 = 2
L_int = 90
Cn = 8.2e-3
Da = 6.07e-3
F = 96485.3321
I_imp = (F * c_max * L) / 3600

t = 0.0
dt = 1e-6

Create Unit Square

In [ ]:
# mesh domain
msh = create_interval(MPI.COMM_WORLD, 220, [0.0, 1.0])
msh2 = create_unit_square(MPI.COMM_WORLD, 96, 96, CellType.triangle)

# mixed 12 elements 6c and 6mu
P1 = element("Lagrange", msh.basix_cell(), 1, dtype=default_real_type)
elements = [P1] * 12
ME = mixed_element(elements)

V_functionspace = functionspace(msh, ME)


print(msh)

In [ ]:
topology, cell_types, geometry = plot.vtk_mesh(msh)
grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)

# Create the plotter
plotter = pyvista.Plotter(window_size=[800, 200])
plotter.add_mesh(grid, show_edges=True, edge_color="black", line_width=2)
plotter.view_xy()
plotter.show()

topology, cell_types, geometry = plot.vtk_mesh(msh2)
grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)

# Create the plotter
plotter = pyvista.Plotter(window_size=[800, 200])
plotter.add_mesh(grid, show_edges=True, edge_color="black", line_width=2)
plotter.view_xy()
plotter.show()

Define Test Functions

In [ ]:
test_functions = TestFunctions(V_functionspace)

q = test_functions[0:6]
v = test_functions[6:12]

Define Variables

In [ ]:
V = Function(V_functionspace)
V0 = Function(V_functionspace)

split_V = ufl.split(V)
c = split_V[0:6]
mu = split_V[6:12]

split_V0 = ufl.split(V0)
c0 = split_V0[0:6]
mu0 = split_V0[6:12]


Define Initial Condition

In [ ]:
c_init = 0.03 + 0.001 * np.random.random()
# c_init = 0.03
mu_init = (math.log(c_init / (1 - c_init))) + omega_a_tilde * (1 - 2 * c_init) + omega_b_tilde * (c_init + c_init) + omega_c_tilde * ((1 - c_init) * c_init + (1 - c_init) * c_init - c_init ** 2)


for i in range(0, 12):
    if i <= 5:
        V.sub(i).interpolate(lambda x: np.full(x.shape[1], c_init))
    else:
        V.sub(i).interpolate(lambda x: np.full(x.shape[1], mu_init))

V.x.scatter_forward()

Compute Chemical Potential and Mobility

In [ ]:
c1 = ufl.variable(c[0])
c2 = ufl.variable(c[1])
c3 = ufl.variable(c[2])
c4 = ufl.variable(c[3])
c5 = ufl.variable(c[4])
c6 = ufl.variable(c[5])

mu1 = ufl.variable(mu[0])
mu2 = ufl.variable(mu[1])
mu3 = ufl.variable(mu[2])
mu4 = ufl.variable(mu[3])
mu5 = ufl.variable(mu[4])
mu6 = ufl.variable(mu[5])

M1 = c1 * (1 - c1)
M2 = c2 * (1 - c2)
M3 = c3 * (1 - c3)
M4 = c4 * (1 - c4)
M5 = c5 * (1 - c5)
M6 = c6 * (1 - c6)

I_imp_tilde = I_imp / i_0

sum_M = (M1 + M2 + M3 + M4 + M5 + M6) / 6.0
sum_M_mu = (M1 * mu1 + M2 * mu2 + M3 * mu3 + M4 * mu4 + M5 * mu5 + M6 * mu6) / 6.0
mu_el = (I_imp_tilde + sum_M_mu) / sum_M

J1_in = Da * M1 * (mu_el - mu1)
J2_in = Da * M2 * (mu_el - mu2)
J3_in = Da * M3 * (mu_el - mu3)
J4_in = Da * M4 * (mu_el - mu4)
J5_in = Da * M5 * (mu_el - mu5)
J6_in = Da * M6 * (mu_el - mu6)

f1 = (ufl.ln(c1 / (1 - c1)) + omega_a_tilde * (1 - 2 * c1) + omega_b_tilde * (c2 + c6) +
      omega_c_tilde * ((1 - c2) * c3 + (1 - c6) * c5 - c2 * c6) + 0)
f2 = (ufl.ln(c2 / (1 - c2)) + omega_a_tilde * (1 - 2 * c2) + omega_b_tilde * (c3 + c1) +
      omega_c_tilde * ((1 - c3) * c4 + (1 - c1) * c6 - c3 * c1) + 0)
f3 = (ufl.ln(c3 / (1 - c3)) + omega_a_tilde * (1 - 2 * c3) + omega_b_tilde * (c4 + c2) +
      omega_c_tilde * ((1 - c4) * c5 + (1 - c2) * c1 - c4 * c2) + 0)
f4 = (ufl.ln(c4 / (1 - c4)) + omega_a_tilde * (1 - 2 * c4) + omega_b_tilde * (c5 + c3) +
      omega_c_tilde * ((1 - c5) * c6 + (1 - c3) * c2 - c5 * c3) + 0)
f5 = (ufl.ln(c5 / (1 - c5)) + omega_a_tilde * (1 - 2 * c5) + omega_b_tilde * (c6 + c4) +
      omega_c_tilde * ((1 - c6) * c1 + (1 - c4) * c3 - c6 * c4) + 0)
f6 = (ufl.ln(c6 / (1 - c6)) + omega_a_tilde * (1 - 2 * c6) + omega_b_tilde * (c1 + c5) +
      omega_c_tilde * ((1 - c1) * c2 + (1 - c5) * c4 - c1 * c5) + 0)


BC for Flux (left)

In [ ]:
from dolfinx.mesh import locate_entities_boundary, meshtags
from ufl import Measure

def left_boundary(x):
    return np.isclose(x[0], 0.0)

fdim = msh.topology.dim - 1
left_facets = locate_entities_boundary(msh, fdim, marker=left_boundary)

# Boundary tag = 1
facet_indices = np.array(left_facets, dtype=np.int32)
sort_idx = np.argsort(facet_indices)
facet_indices = facet_indices[sort_idx]
facet_markers = np.full_like(facet_indices, 1, dtype=np.int32)

facet_tags = meshtags(msh, fdim, facet_indices, facet_markers)

# Define the integration with tag as ds
ds = Measure("ds", domain=msh, subdomain_data=facet_tags)

Weak Form of Equations

In [ ]:
F_C1 = (
    ufl.inner(c1, q[0]) * ufl.dx
    - ufl.inner(c0[0], q[0]) * ufl.dx
    + dt * M1 * ufl.inner(ufl.grad(mu1), ufl.grad(q[0])) * ufl.dx
    - dt * J1_in * q[0] * ds(1)

)
F_C2 = (
    ufl.inner(c2, q[1]) * ufl.dx
    - ufl.inner(c0[1], q[1]) * ufl.dx
    + dt * M2 * ufl.inner(ufl.grad(mu2), ufl.grad(q[1])) * ufl.dx
    - dt * J2_in * q[1] * ds(1)
)
F_C3 = (
    ufl.inner(c3, q[2]) * ufl.dx
    - ufl.inner(c0[2], q[2]) * ufl.dx
    + dt * M3 * ufl.inner(ufl.grad(mu3), ufl.grad(q[2])) * ufl.dx
    - dt * J3_in * q[2] * ds(1)
)
F_C4 = (
    ufl.inner(c4, q[3]) * ufl.dx
    - ufl.inner(c0[3], q[3]) * ufl.dx
    + dt * M4 * ufl.inner(ufl.grad(mu4), ufl.grad(q[3])) * ufl.dx
    - dt * J4_in * q[3] * ds(1)
)
F_C5 = (
    ufl.inner(c5, q[4]) * ufl.dx
    - ufl.inner(c0[4], q[4]) * ufl.dx
    + dt * M5 * ufl.inner(ufl.grad(mu5), ufl.grad(q[4])) * ufl.dx
    - dt * J5_in * q[4] * ds(1)
)
F_C6 = (
    ufl.inner(c6, q[5]) * ufl.dx
    - ufl.inner(c0[5], q[5]) * ufl.dx
    + dt * M6 * ufl.inner(ufl.grad(mu6), ufl.grad(q[5])) * ufl.dx
    - dt * J6_in * q[5] * ds(1)
)

F_mu1 = (
    ufl.inner(mu1, v[0]) * ufl.dx
    - ufl.inner(f1, v[0]) * ufl.dx
    + Cn ** 2 * ufl.inner(ufl.grad(c1), ufl.grad(v[0])) * ufl.dx
)
F_mu2 = (
    ufl.inner(mu2, v[1]) * ufl.dx
    - ufl.inner(f2, v[1]) * ufl.dx
    + Cn ** 2 * ufl.inner(ufl.grad(c2), ufl.grad(v[1])) * ufl.dx
)
F_mu3 = (
    ufl.inner(mu3, v[2]) * ufl.dx
    - ufl.inner(f3, v[2]) * ufl.dx
    + Cn ** 2 * ufl.inner(ufl.grad(c3), ufl.grad(v[2])) * ufl.dx
)
F_mu4 = (
    ufl.inner(mu4, v[3]) * ufl.dx
    - ufl.inner(f4, v[3]) * ufl.dx
    + Cn ** 2 * ufl.inner(ufl.grad(c4), ufl.grad(v[3])) * ufl.dx
)
F_mu5 = (
    ufl.inner(mu5, v[4]) * ufl.dx
    - ufl.inner(f5, v[4]) * ufl.dx
    + Cn ** 2 * ufl.inner(ufl.grad(c5), ufl.grad(v[4])) * ufl.dx
)
F_mu6 = (
    ufl.inner(mu6, v[5]) * ufl.dx
    - ufl.inner(f6, v[5]) * ufl.dx
    + Cn ** 2 * ufl.inner(ufl.grad(c6), ufl.grad(v[5])) * ufl.dx
)

F = F_C1 + F_C2 + F_C3 + F_C4 + F_C5 + F_C6 + \
    F_mu1 + F_mu2 + F_mu3 + F_mu4 + F_mu5 + F_mu6

Solver set

In [ ]:
use_superlu = PETSc.IntType == np.int64  # or PETSc.ScalarType == np.complex64
sys = PETSc.Sys()  # type: ignore
if sys.hasExternalPackage("mumps") and not use_superlu:
    linear_solver = "mumps"
elif sys.hasExternalPackage("superlu_dist"):
    linear_solver = "superlu_dist"
else:
    linear_solver = "petsc"

petsc_options = {
    # "snes_type": "newtonls",
    "snes_type": "ngmres",
    "snes_linesearch_type": "none",
    "snes_stol": np.sqrt(np.finfo(default_real_type).eps) * 1e-2,
    "snes_atol": 1e-6,
    "snes_rtol": 1e-6,
    "ksp_type": "gmres",
    "pc_type": "none",
    "pc_factor_mat_solver_type": linear_solver,
    "snes_monitor": None,
}
problem = NonlinearProblem(
    F, V, petsc_options_prefix="demo_cahn-hilliard_", petsc_options=petsc_options
)

Output files

In [ ]:
# out_folder = Path("multilayerCH")
# out_folder.mkdir(parents=True, exist_ok=True)
# file = XDMFFile(MPI.COMM_WORLD, out_folder / "output.xdmf", "w")  # Output file
# file.write_mesh(msh)

Run Time

In [ ]:
T = 3.6e9 * dt

Solver

In [ ]:
c_out = []
for i in range(6):
    c_t = V.sub(i).collapse()
    c_t.name = f"c{i+1}"
    c_out.append(c_t)

V0.x.array[:] = V.x.array
step = 0

snapshot_times = [0.1 * T, 0.5 * T, 0.9 * T]
snapshot_idx = 0

while t < T:
    t += dt
    _ = problem.solve()
    converged_reason = problem.solver.getConvergedReason()
    assert converged_reason > 0
    num_iterations = problem.solver.getIterationNumber()
    print(f"Step {step}: {converged_reason=} {num_iterations=}")

    V0.x.array[:] = V.x.array

    # if step % 100 == 0:
    #     for i in range(6):
    #         c_out[i].x.array[:] = V.sub(i).collapse().x.array[:]
    #         file.write_function(c_out[i], t)
    #     file.flush()
    step += 1


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.style.use('default')

rows = []
x_sorted = None

for i in range(6):
    c_func = V.sub(i).collapse()
    dof_coords = c_func.function_space.tabulate_dof_coordinates()[:, 0]
    sort_idx = np.argsort(dof_coords)

    if i == 0:
        x_sorted = dof_coords[sort_idx]

    c_vals = c_func.x.array[sort_idx]
    rows.append(c_vals)

concentration_matrix = np.vstack(rows)

fig, ax = plt.subplots(figsize=(10, 4), facecolor='white')
ax.set_facecolor('white')


im = ax.imshow(concentration_matrix, cmap='gray', aspect='auto', origin='lower',
               extent=[x_sorted[0], x_sorted[-1], 0.5, 6.5])

cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Concentration', fontsize=12, color='black')
cbar.ax.yaxis.set_tick_params(color='black', labelcolor='black')

ax.set_yticks([1, 2, 3, 4, 5, 6])
ax.set_yticklabels(['Layer 1', 'Layer 2', 'Layer 3', 'Layer 4', 'Layer 5', 'Layer 6'],
                   color='black', fontsize=12, fontweight='bold')

ax.tick_params(axis='x', colors='black', labelsize=11)
ax.set_xlabel("Position (x)", fontsize=12, color='black')
ax.set_ylabel("Layers", fontsize=12, color='black')
ax.set_title(f"Multi-layer Concentration Map at t = {t:.6f}", fontsize=14, color='black')

plt.tight_layout()
plt.show()